# 01 - Load the Sepsis event log

Confirms the raw XES log loads correctly and checks its shape, columns and sample rows against `docs/data_dictionary.md`.

In [1]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
sys.path.append(str(Path.cwd().parent / 'src'))

from load_data import load_event_log, summarize, CASE_ID, ACTIVITY, TIMESTAMP, RESOURCE

log = load_event_log()

parsing log, completed traces ::   0%|          | 0/1050 [00:00<?, ?it/s]

## Shape and headline numbers

In [2]:
print('Shape (rows, columns):', log.shape)
summarize(log)

Shape (rows, columns): (15214, 32)


{'events': 15214,
 'cases': 1050,
 'activities': 16,
 'columns': 32,
 'start': Timestamp('2013-11-07 07:18:29+0000', tz='UTC'),
 'end': Timestamp('2015-06-05 10:25:11+0000', tz='UTC')}

## Sample rows
The three fields process mining needs: case ID, activity, timestamp (plus the department).

In [3]:
log[[CASE_ID, ACTIVITY, TIMESTAMP, RESOURCE, 'lifecycle:transition']].head(10)

,case:concept:name,concept:name,time:timestamp,org:group,lifecycle:transition
0,A,ER Registration,2014-10-22 09:15:41+00:00,A,complete
1,A,Leucocytes,2014-10-22 09:27:00+00:00,B,complete
2,A,CRP,2014-10-22 09:27:00+00:00,B,complete
3,A,LacticAcid,2014-10-22 09:27:00+00:00,B,complete
4,A,ER Triage,2014-10-22 09:33:37+00:00,C,complete
5,A,ER Sepsis Triage,2014-10-22 09:34:00+00:00,A,complete
6,A,IV Liquid,2014-10-22 12:03:47+00:00,A,complete
7,A,IV Antibiotics,2014-10-22 12:03:47+00:00,A,complete
8,A,Admission NC,2014-10-22 12:13:19+00:00,D,complete
9,A,CRP,2014-10-24 07:00:00+00:00,B,complete


## Columns and data types

In [4]:
log.dtypes.to_frame('dtype')

,dtype
InfectionSuspected,object
org:group,str
DiagnosticBlood,object
DisfuncOrg,object
SIRSCritTachypnea,object
Hypotensie,object
SIRSCritHeartRate,object
Infusion,object
DiagnosticArtAstrup,object
concept:name,str


## Missing values
Most case attributes (Age, diagnostic flags, SIRS criteria) are only recorded on the registration event, so they are mostly empty at event level. This is handled in the cleaning step.

In [5]:
(log.isna().mean().mul(100).round(1).sort_values(ascending=False).to_frame('% missing'))

,% missing
Diagnose,94.8
InfectionSuspected,93.1
DisfuncOrg,93.1
DiagnosticBlood,93.1
SIRSCritHeartRate,93.1
Hypotensie,93.1
Infusion,93.1
SIRSCritTachypnea,93.1
DiagnosticOther,93.1
Age,93.1


## Activities and departments

In [6]:
log[ACTIVITY].value_counts().to_frame('events')

,events
concept:name,
Leucocytes,3383
CRP,3262
LacticAcid,1466
Admission NC,1182
ER Triage,1053
ER Registration,1050
ER Sepsis Triage,1049
IV Antibiotics,823
IV Liquid,753


In [7]:
log[RESOURCE].value_counts().to_frame('events')

,events
org:group,
B,8111
A,3462
C,1053
E,782
?,294
F,216
L,213
O,186
G,148


## Events per case

In [8]:
log.groupby(CASE_ID).size().describe().round(1).to_frame('events per case')

,events per case
count,1050.0
mean,14.5
std,11.5
min,3.0
25%,9.0
50%,13.0
75%,16.0
max,185.0


## Findings
- The log has 1,050 cases, 15,214 events and 16 activities, matching the dataset documentation.
- All 32 columns match the data dictionary.
- Events span Nov 2013 to Jun 2015.
- `org:group` holds anonymised department codes (A, B, C, ...), not names.
- Case attributes are about 93% empty at event level, so they need to be carried per case in the cleaning step.